# SignalCortex — Batch Training on Colab

## Setup Guide

1. **Upload files to Google Drive:**
   - Upload `SignalCortex.zip` to `My Drive/SignalCortex/`
   - Upload all parquet files to `My Drive/SignalCortex/parquet/`

2. **Runtime config:**
   - Go to Runtime > Change runtime type > **A100 GPU**
   - If A100 is unavailable, V100 or T4 also work (slower)

3. **Run cells in order** — Cell 1 mounts Drive, Cell 2 extracts files, Cell 3 installs deps, Cell 4 runs the batch

In [ ]:
# Cell 1: Mount Google Drive & Configure
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURE THESE ===
DRIVE_BASE = '/content/drive/MyDrive/SignalCortex'
ZIP_PATH = f'{DRIVE_BASE}/SignalCortex.zip'
PARQUET_DIR = f'{DRIVE_BASE}/parquet'

# Batch config
EXPERIMENT = 'fast_iteration'
TAG = 'raw_v2'
PW_VALUES = '7.0,8.0,9.0'
LR_VALUES = '0.0003'
# =======================

In [ ]:
# Cell 2: Extract project files
import os, shutil

WORK_DIR = '/content/SignalCortex'
os.chdir('/content')
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

!unzip -q -o {ZIP_PATH} -d {WORK_DIR}
os.chdir(WORK_DIR)

# Verify structure
!ls scripts/

# Verify parquet files exist
parquet_files = [f for f in os.listdir(PARQUET_DIR) if f.endswith('.parquet')]
print(f'Working dir: {os.getcwd()}')
print(f'Parquet files: {len(parquet_files)}')
for f in sorted(parquet_files):
    size = os.path.getsize(os.path.join(PARQUET_DIR, f)) / 1024 / 1024
    print(f'  {f} ({size:.1f} MB)')

# Show GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 3: Install dependencies
!pip install -q tensorboard tqdm pyyaml scikit-learn openpyxl

In [ ]:
# Cell 4: Run batch training + backtest
!python scripts/batch_train.py \
    --config configs/experiments/{EXPERIMENT}.yaml \
    --tag {TAG} \
    --pw {PW_VALUES} \
    --lr {LR_VALUES} \
    --parquet {PARQUET_DIR}

In [ ]:
# Cell 5: Copy results to Drive for persistence
import shutil, glob

output_dir = 'outputs/fast_iteration'
drive_output = f'{DRIVE_BASE}/results/{TAG}'
os.makedirs(drive_output, exist_ok=True)

# Copy checkpoints
for f in glob.glob(f'{output_dir}/best_*.pt'):
    dst = os.path.join(drive_output, os.path.basename(f))
    shutil.copy2(f, dst)
    print(f'Copied: {os.path.basename(f)}')

# Copy backtest Excel folders
for d in glob.glob(f'{output_dir}/backtest_*'):
    if os.path.isdir(d):
        dst = os.path.join(drive_output, os.path.basename(d))
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(d, dst)
        print(f'Copied folder: {os.path.basename(d)}')

print(f'\nAll results saved to: {drive_output}')

In [ ]:
# Cell 6: List all outputs
for root, dirs, files in os.walk('outputs'):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f'{path} ({size:.1f} MB)')